In [1]:
%reload_ext autoreload
%autoreload 2

In [30]:
import mcts
import tm
import heuristic
import task_generators
import numpy as np
import training
import torch
from torch import nn
from torch import optim
from time import time

In [116]:
tm_state_base = tm.TMState()
heuristic_model = heuristic.HeuristicModel(tm_state_base)

In [20]:
training_samples = training.generate_training_samples(heuristic_model.rollout_function, task_generators.copy_task_generator)

Generating data for target: tensor([1., 1., 0., 1., 1.], dtype=torch.float64)
Node 2 completed
Generated samples for game 0/10
Generating data for target: tensor([1., 1., 1., 0., 1., 1., 1.], dtype=torch.float64)
Node 2 completed
Generated samples for game 1/10
Generating data for target: tensor([1., 1., 0., 1., 1.], dtype=torch.float64)
Node 2 completed
Node 3 completed
Generated samples for game 2/10
Generating data for target: tensor([1., 1., 1., 0., 1., 1., 1.], dtype=torch.float64)
Node 2 completed
Node 3 completed
Node 4 completed
Node 5 completed
Generated samples for game 3/10
Generating data for target: tensor([1., 0., 1.], dtype=torch.float64)
Node 2 completed
Node 3 completed
Node 4 completed
Node 5 completed
Node 6 completed
Node 7 completed
Node 8 completed
Node 9 completed
Node 10 completed
Node 11 completed
Node 12 completed
Node 13 completed
Generated samples for game 4/10
Generating data for target: tensor([1., 1., 1., 0., 1., 1., 1.], dtype=torch.float64)
Node 2 compl

In [24]:
training.save_training_data(training_samples, "0_epochs_copy")

In [26]:
test = torch.load("./training_data/0_epochs_copy")

In [117]:
def train_heuristic(heuristic, training_data, epochs = 256):
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.SGD(heuristic.parameters(), lr=0.5)
    start_time = time()

    for i in range(epochs):
        running_loss = 0
        training_samples = 0

        for state, action in training_data[:1]:
            optimizer.zero_grad()
            output_action = heuristic(state)
            loss = loss_function(output_action, action.to(torch.float))
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            training_samples += 1
        
        print(f"Epoch {i}/{epochs} - Loss {running_loss/training_samples} - Time {time() - start_time}s")

In [125]:
train_heuristic(heuristic_model.model, training_samples)

Epoch 0/256 - Loss 3.865372657775879 - Time 0.00299835205078125s
Epoch 1/256 - Loss 3.865372657775879 - Time 0.005998373031616211s
Epoch 2/256 - Loss 3.865372657775879 - Time 0.008997440338134766s
Epoch 3/256 - Loss 3.865372657775879 - Time 0.010997772216796875s
Epoch 4/256 - Loss 3.865372657775879 - Time 0.01299738883972168s
Epoch 5/256 - Loss 3.865372657775879 - Time 0.015999555587768555s
Epoch 6/256 - Loss 3.865372657775879 - Time 0.0189974308013916s
Epoch 7/256 - Loss 3.865372657775879 - Time 0.020998001098632812s
Epoch 8/256 - Loss 3.865372657775879 - Time 0.02399730682373047s
Epoch 9/256 - Loss 3.865372657775879 - Time 0.02599811553955078s
Epoch 10/256 - Loss 3.865372657775879 - Time 0.027997970581054688s
Epoch 11/256 - Loss 3.865372657775879 - Time 0.03099822998046875s
Epoch 12/256 - Loss 3.865372657775879 - Time 0.03299832344055176s
Epoch 13/256 - Loss 3.865372657775879 - Time 0.03599715232849121s
Epoch 14/256 - Loss 3.865372657775879 - Time 0.03799724578857422s
Epoch 15/256 - 

In [128]:
with torch.no_grad():
    hard_model = nn.Sequential(heuristic_model.model[0:-1])
    for state, action in training_samples[:1]:
        output = hard_model(state)
        print(torch.argmax(output), torch.max(output))
        print(torch.argmax(action))


tensor(7) tensor(16.7395)
tensor(7)
